In [17]:
from pathlib import Path
import re
import pandas as pd
import nltk
from sklearn.feature_extraction.text import TfidfVectorizer  # ← Cambio aquí
from nltk.corpus import stopwords

# Descargar stopwords
nltk.download('stopwords')
stopwords_es = stopwords.words("spanish")

# 1. Cargar documentos
directorio_con_datos = "../buscador/all_pages_clean/"
ruta = Path(directorio_con_datos)

print("Cargando documentos...")
archivos = list(ruta.glob("*.txt"))

def extraer_numero(archivo):
    numeros = re.findall(r'(\d+)', archivo.stem)
    return int(numeros[0]) if numeros else 0

archivos.sort(key=extraer_numero)

documentos = []
nombres_archivos = []
for archivo in archivos:
    try:
        with open(archivo, 'r', encoding='utf-8') as f:
            contenido = f.read()
            if contenido.strip():
                documentos.append(contenido)
                nombres_archivos.append(archivo.name)
    except Exception as e:
        print(f"Error leyendo {archivo.name}: {e}")

print(f"✅ Cargados {len(documentos)} documentos")

# 2. Crear vectorizador TF-IDF (¡NO CountVectorizer!)
print("\nConstruyendo matriz TF-IDF...")
vectorizador = TfidfVectorizer(
    input='content',
    encoding='utf-8',
    decode_error='ignore',
    strip_accents='ascii',     # Normaliza acentos
    lowercase=True,            # Minúsculas
    stop_words=stopwords_es,   # Stopwords en español
    ngram_range=(1, 2),        # Unigramas y bigramas
    max_df=0.8,                # Ignora palabras muy comunes
    min_df=2,                  # Ignora palabras muy raras
    max_features=10000,        # Límite de características
    token_pattern=r'(?u)\b\w+\b',
    use_idf=True,              # ← Usa IDF (por defecto True)
    smooth_idf=True,           # ← Suavizado para evitar división por cero
    norm='l2'                  # ← Normalización Euclidean
)

# Crear matriz TF-IDF
matriz_tfidf = vectorizador.fit_transform(documentos)

# Obtener vocabulario
vocabulario = vectorizador.get_feature_names_out()

# 3. Resultados
print(f"\n--- RESULTADOS TF-IDF ---")
print(f"Documentos: {len(documentos)}")
print(f"Características: {len(vocabulario)}")
print(f"Matriz shape: {matriz_tfidf.shape}")

if len(vocabulario) > 0:
    print(f"\nEjemplo de características: {vocabulario[:20]}")
    
    # Mostrar pesos TF-IDF del primer documento
    print(f"\nPesos TF-IDF para el documento {nombres_archivos[0]}:")
    pesos_primer_doc = matriz_tfidf[0].toarray()[0]
    top_indices = pesos_primer_doc.argsort()[-10:][::-1]
    
    for idx in top_indices:
        if pesos_primer_doc[idx] > 0:
            print(f"  {vocabulario[idx]}: {pesos_primer_doc[idx]:.4f}")
    
    # Guardar a CSV
    df_tfidf = pd.DataFrame(
        matriz_tfidf.toarray(),
        columns=vocabulario,
        index= [int(name[:-4]) for name in nombres_archivos]
    )
    
    df_tfidf.to_csv('matriz_tfidf.csv', encoding='utf-8')
    print(f"\n✅ Archivo CSV guardado: matriz_tfidf.csv")
    print(f"   Dimensiones: {df_tfidf.shape}")

[nltk_data] Downloading package stopwords to
[nltk_data]     /home/juancho/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
/home/juancho/.local/lib/python3.10/site-packages/sklearn/feature_extraction/text.py:408: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['eramos', 'estabamos', 'estais', 'estan', 'estara', 'estaran', 'estaras', 'estare', 'estareis', 'estaria', 'estariais', 'estariamos', 'estarian', 'estarias', 'esteis', 'esten', 'estes', 'estuvieramos', 'estuviesemos', 'fueramos', 'fuesemos', 'habeis', 'habia', 'habiais', 'habiamos', 'habian', 'habias', 'habra', 'habran', 'habras', 'habre', 'habreis', 'habria', 'habriais', 'habriamos', 'habrian', 'habrias', 'hayais', 'hubieramos', 'hubiesemos', 'mas', 'mia', 'mias', 'mio', 'mios', 'seais', 'sera', 'seran', 'seras', 'sere', 'sereis', 'seria', 'seriais', 'seriamos', 'serian', 'serias', 'si', 'tambien', 'tendra', 'tendran', 'tendras', 'tendre', 

Cargando documentos...
✅ Cargados 548 documentos

Construyendo matriz TF-IDF...

--- RESULTADOS TF-IDF ---
Documentos: 548
Características: 10000
Matriz shape: (548, 10000)

Ejemplo de características: ['0' '0 0' '0 00' '0 01' '0 05' '0 1' '0 10' '0 100' '0 2' '0 3' '0 4'
 '0 5' '0 6' '0 7' '0 8' '0 9' '00' '00 00' '00 m' '000']

Pesos TF-IDF para el documento 0000.txt:
  ley: 0.4044
  editar: 0.3792
  wikipedia: 0.3718
  leyes: 0.1854
  norma: 0.1442
  derecho: 0.1351
  consultado: 0.1066
  mover: 0.0984
  barra: 0.0973
  27 febrero: 0.0944

✅ Archivo CSV guardado: matriz_tfidf.csv
   Dimensiones: (548, 10000)
